# 04 - Explainable AI (XAI) with SHAP & Clinical Point Alignment

## Clinical Motivation
Black-box machine learning models cannot be safely deployed in clinical environments without transparent rationale. 
Physicians must understand:
1. **Which patient factors drove this prediction?** (e.g., severe hypertension vs acute glucose spikes).
2. **Global model drivers**: What are the most influential clinical features across the patient population?
3. **Auditability**: How does the machine learning attribution align with guideline-based clinical points (AHA/ACC, Framingham risk rules)?

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
shap.initjs()

## 1. Load Model Pipeline & Sample Test Cohort

In [ ]:
from src.schema_mapping import PROCESSED_DIR
from src.preprocessing import ClinicalDataPreprocessor, EndToEndRiskPipeline

model_path = PROJECT_ROOT / "models" / "best_model.joblib"
pipeline = joblib.load(model_path)
print("Loaded EndToEndRiskPipeline successfully.")

test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
TARGET_COL = "target_risk"
SOURCE_COL = "source_dataset"

feature_cols = [c for c in test_df.columns if c not in [TARGET_COL, SOURCE_COL]]
sample_df = test_df[feature_cols].sample(n=500, random_state=42)

# Preprocess through the fitted transformer
X_transformed = pipeline.preprocessor.transform(sample_df)
feature_names = getattr(pipeline.preprocessor, "feature_names_", [f"f{i}" for i in range(X_transformed.shape[1])])

print(f"Sample transformed: {X_transformed.shape[0]} patients × {X_transformed.shape[1]} engineered features")

## 2. SHAP TreeExplainer Initialization
We instantiate a `shap.TreeExplainer` on the fitted XGBoost tree ensemble.

In [ ]:
explainer = shap.TreeExplainer(pipeline.classifier)
shap_values = explainer.shap_values(X_transformed)

if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_vals_pos = shap_values[1]
else:
    shap_vals_pos = shap_values

print(f"SHAP values computed: {shap_vals_pos.shape}")

## 3. Global Explainability: Feature Importance Bar Plot
Ranks features by their mean absolute SHAP attribution across the cohort.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_vals_pos, X_transformed, feature_names=feature_names, plot_type="bar", show=False)
plt.title("Global Feature Importance (Mean |SHAP Value|)", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## 4. Global Explainability: Beeswarm Plot
Shows both magnitude and direction of feature effect (e.g. high resting BP increases risk, high heart rate variability decreases risk).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_vals_pos, X_transformed, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm Summary Plot: Impact on Adverse Clinical Risk", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

## 5. Local Patient Case Study: High-Risk vs Low-Risk Profiles
Let's inspect two distinct patient cases to see individualized explanations:

In [ ]:
from src.rule_based_risk import calculate_rule_based_risk

# Patient 1: High Risk Profile
patient_high = {
    "age": 68,
    "sex": 1,
    "resting_bp": 178,
    "cholesterol": 290,
    "fasting_blood_sugar": 185,
    "max_heart_rate": 115,
    "bmi": 34.2,
    "smoking": 1,
    "diabetes": 1,
    "hypertension": 1
}

# Patient 2: Low Risk Profile
patient_low = {
    "age": 28,
    "sex": 0,
    "resting_bp": 114,
    "cholesterol": 165,
    "fasting_blood_sugar": 85,
    "max_heart_rate": 170,
    "bmi": 21.5,
    "smoking": 0,
    "diabetes": 0,
    "hypertension": 0
}

# Evaluate with Rule Engine
rule_high = calculate_rule_based_risk(patient_high)
rule_low = calculate_rule_based_risk(patient_low)

print("=" * 60)
print("CASE 1: HIGH-RISK CLINICAL PROFILE")
print("=" * 60)
print(f"Rule Engine Score : {rule_high['total_points']} points -> Risk Band: {rule_high['risk_band']}")
print("Contributing Clinical Factors:")
for pt, desc in rule_high['point_breakdown']:
    print(f"  * {desc} (+{pt} pts)")

print("\n" + "=" * 60)
print("CASE 2: LOW-RISK CLINICAL PROFILE")
print("=" * 60)
print(f"Rule Engine Score : {rule_low['total_points']} points -> Risk Band: {rule_low['risk_band']}")
if not rule_low['point_breakdown']:
    print("  * No adverse clinical thresholds flagged.")

## 6. Patient-Level SHAP Waterfall / Force Explanation
Let's compute SHAP values for the high-risk patient:

In [ ]:
patient_df = pd.DataFrame([patient_high])
pt_transformed = pipeline.preprocessor.transform(patient_df)
pt_shap = explainer.shap_values(pt_transformed)

if isinstance(pt_shap, list) and len(pt_shap) == 2:
    pt_shap = pt_shap[1]

# Rank features for this individual patient
pt_contribs = pd.DataFrame({
    "Feature": feature_names,
    "SHAP Value": pt_shap[0]
}).sort_values(by="SHAP Value", ascending=False)

print("Top 5 Risk-Elevating Factors for Patient 1:")
print(pt_contribs.head(5).to_string(index=False))

print("\nTop Protective Factors for Patient 1:")
print(pt_contribs.tail(3).to_string(index=False))

## 7. Conclusions
1. **Model Transparency**: Individual SHAP values give physicians unambiguous reasons behind algorithmic risk estimates.
2. **Clinical Concordance**: Factors identified by gradient-boosted SHAP values closely align with established medical guidelines (AHA/ACC, Framingham, ADA).
3. **Safety Assurance**: The deterministic rule engine provides an immediate, auditable clinical point baseline, while SHAP offers deep non-linear insight.